In [0]:
from pyspark.sql.functions import col, avg, max, round, lag, sum
from pyspark.sql.window import Window

daily_df = spark.table("weather_project.silver.daily_clean")


# Нужно добавить:
# -- Lag temperature (1 day)
# -- Rolling avg temperature (7 days)
# -- Rain last 3 days indicator 

# Создаем оконные функции
window_city = Window.partitionBy("city_name").orderBy("date")
window_3d = Window.partitionBy("city_name").orderBy("date").rowsBetween(-3, -1)
window_7d = Window.partitionBy("city_name").orderBy("date").rowsBetween(-6, 0)

features_df = daily_df.select("city_name", "date", "temperature_avg", "has_rain") \
    .withColumn("lag_temperature_1d",round(lag("temperature_avg", 1).over(window_city), 2)) \
    .withColumn("rolling_avg_temperature_7d",round(avg("temperature_avg").over(window_7d), 2)) \
    .withColumn("rain_last_3d", sum(col("has_rain").cast("int")).over(window_3d) > 0)

features_df.write.mode("overwrite").saveAsTable("weather_project.gold.ml_features")

print(f"ml_features: {features_df.count()} rows")
display(features_df.limit(25)) 